# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)

import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

df = pd.read_csv('./customers.csv')

df.head()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column

print(df.isna().sum()) #shows missing values per column
print()
print(df['region'].unique()) #shows all of the unique names in the column region
print()
print(df.describe())

df['region'] = df['region'].str.strip().str.title() #standardizing the region column ('east' to 'East')

df.loc[df['income'] <= 0, 'income'] = np.nan #makes all the values below 0 as missing

df['income'] = df['income'].fillna(df['income'].median())

print()
print(df.isna().sum())
print(df['region'].unique())
print()
print(df.describe())



customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64

<StringArray>
['West', 'South', 'East', 'east']
Length: 4, dtype: str

       customer_id        age      income   spending
count    40.000000  40.000000   38.000000  40.000000
mean     20.500000  40.925000   83.007895  54.520000
std      11.690452  11.474358   27.628563   9.377912
min       1.000000  21.000000  -50.000000  36.000000
25%      10.750000  32.500000   72.975000  48.750000
50%      20.500000  41.000000   84.700000  53.650000
75%      30.250000  51.000000   98.500000  62.650000
max      40.000000  59.000000  114.500000  74.700000

customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East']
Length: 3, dtype: str

       customer_id        age      income   spending
count    40.000000  40.000000   40.000000  40.000000
mean     20.500000  40.925000   86.460000  54.52000

**Explain each issue:** what you found and how you handled it.

*Your answer:*

For the first issue, there were two "East" values in the column region. I standardized it by using the strip() and title() to change 'east' to "East". For the second issue, there was an income that was -50, and that is not possible, so you first label it as missing, which is what I did. After that, I filled in the missing value with the median because it is safer than using mean as there are 40 rows. 

## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types: income = ?, spending = ?
# tool: ?
# hint: two continuous variables moving together

# inome and spending are both continuous (numeric)
# the tool you use is Pearson correlation

r,p = stats.pearsonr(df['income'], df['spending'])

print(f"The p-value is: {p:3f}")
print()
print(f"The r-value is {r:3f}")

The p-value is: 0.000000

The r-value is 0.717222


*Your answer:*
Income and spending are both continuous (numeric) and the tool that I used is Pearson correlation. Customers who earn more tend to spend more as the r-value is high.


### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = ?, subscribed = ?
# tool: ?
# hint: two categorical variables -> build a table with pd.crosstab, then test it

# region and subscribed are both categorical
# Chi-squared test of independence

table = pd.crosstab(df['region'], df['subscribed']) # creating the table
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"The chi2 value is: {chi2:.3f}")
print(f"The Degree of Freedom is: {dof}")
print(f"The p-value is: {p:.4f}")


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1
The chi2 value is: 13.162
The Degree of Freedom is: 2
The p-value is: 0.0014


*Your answer:*
Region and subscribed are both categorical and the tool that I used was chi-squared test of independence. A customer's region is related to whether they subscribed because the East region is omre likely to subscribe than the South and West.

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value

# spending is a continuous outcome, while income and age are continuous predictors.
# the tool that I am using is Multiple Linear Regression

X = sm.add_constant(df[['income', 'age']])
y = df['spending']

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        15:58:29   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:*


### Q4. Which region has the most customers, and what is the average income overall?

In [6]:
# hint: count a categorical column
# take the mean of a numeric column

print(df['region'].value_counts())
print(f"The average income is: {df['income'].mean() * 1000}")

region
East     21
West     10
South     9
Name: count, dtype: int64
The average income is: 86460.0


*Your answer:*
The East Region has the most customers and the average income overall is 86,460.

## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:*
The way I knew what tool to use each time was based on the data types of the values that I was working with. For the correlation question, both values were continuous. The chi-squared question had values that were both categorical. The multiple regression had one value that was a continuous outcome and the other was a continuous predictor. For the last question, there was only one variable.

**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:*

It does tell the company that higher income causes higher spending because correlation basically shows when one variable increases, the other one as well. A correlation doesn't necessarily show a cause and effect relationship.


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [7]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

expected_df = pd.DataFrame(expected, index = table.index, columns = table.columns)
print(expected_df)

print("All expected counts >=5,", (expected >= 5).all())


subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0
All expected counts >=5, False


*Your answer:*
No, they do not all meet the assumption we learned in the chi-squared module, and the company can't trust that result because not all values are under 5 which makes the test mathematically unstable.


### Push your code to GitHub